# mlX 2.0 Regression Challenge: Predict the Hype

Clean, report-ready notebook with two different regression approaches and two submission files.


## 1. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

RANDOM_STATE = 42
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_SUB_PATH = "sample_submission.csv"

APPROACH_NOTES = {
    "Model 1: LinearRegression": {
        "why": "Strong baseline: fast, interpretable, and establishes a linear reference point.",
        "how": "Fits a linear relationship between engineered features and target after imputation, scaling, and one-hot encoding.",
        "drawbacks": "Cannot naturally model complex non-linear feature interactions; may underfit high-variance patterns.",
    },
    "Model 2: GradientBoosting": {
        "why": "Captures non-linear interactions and often improves RMSE in tabular regression tasks.",
        "how": "Builds an additive ensemble of shallow regression trees that iteratively correct previous residual errors.",
        "drawbacks": "More computationally expensive and less interpretable than linear models; can overfit if not tuned.",
    },
}


## 2. Utility Functions

In [ ]:
def print_initial_exploration(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    print("\n" + "=" * 80)
    print("STEP 1: DATA LOADING & INITIAL EXPLORATION")
    print("=" * 80)
    print(f"Train shape: {train_df.shape}")
    print(f"Test shape : {test_df.shape}")
    print("\nTrain columns:")
    print(train_df.columns.tolist())
    print("\nMissing values in train:")
    print(train_df.isna().sum().sort_values(ascending=False).head(20))
    print("\nMissing values in test:")
    print(test_df.isna().sum().sort_values(ascending=False).head(20))
    print("\nData types in train:")
    print(train_df.dtypes)


def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    return a / b.replace(0, np.nan)


def add_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "publication_timestamp" in out.columns:
        out["publication_timestamp"] = pd.to_datetime(out["publication_timestamp"], errors="coerce")
        out["pub_year"] = out["publication_timestamp"].dt.year
        out["pub_month"] = out["publication_timestamp"].dt.month
        out["pub_day"] = out["publication_timestamp"].dt.day
        out["pub_weekday"] = out["publication_timestamp"].dt.weekday
        out = out.drop(columns=["publication_timestamp"])

    energy_cols = [c for c in out.columns if "emotional_charge" in c]
    dance_cols = [c for c in out.columns if "groove_efficiency" in c]
    tempo_like_cols = [c for c in out.columns if "beat_frequency" in c or "tempo" in c]

    if energy_cols:
        out["avg_energy"] = out[energy_cols].mean(axis=1)
    if dance_cols:
        out["avg_danceability"] = out[dance_cols].mean(axis=1)
    if tempo_like_cols:
        out["avg_tempo"] = out[tempo_like_cols].mean(axis=1)
        out["tempo_range"] = out[tempo_like_cols].max(axis=1) - out[tempo_like_cols].min(axis=1)

    if "avg_energy" in out.columns and "avg_danceability" in out.columns:
        out["energy_to_danceability"] = safe_divide(out["avg_energy"], out["avg_danceability"]).replace([np.inf, -np.inf], np.nan)

    drop_cols = [
        "track_identifier",
        "composition_label_0",
        "composition_label_1",
        "composition_label_2",
    ]
    drop_cols = [c for c in drop_cols if c in out.columns]
    if drop_cols:
        out = out.drop(columns=drop_cols)

    return out


def build_preprocessor(X: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=numeric_steps), numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )


def evaluate_model(model_name: str, y_true: pd.Series, y_pred: np.ndarray) -> dict:
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return {"Model": model_name, "RMSE": rmse, "MAE": mae, "R2": r2, "MAPE": mape}


def find_best_blend_weight(y_true: pd.Series, pred1: np.ndarray, pred2: np.ndarray) -> tuple[float, float]:
    best_w = 0.0
    best_rmse = float("inf")
    for w in np.linspace(0.0, 1.0, 21):
        blended = w * pred1 + (1.0 - w) * pred2
        rmse = np.sqrt(mean_squared_error(y_true, blended))
        if rmse < best_rmse:
            best_rmse = rmse
            best_w = float(w)
    return best_w, best_rmse


## 3. Load Data and Initial Exploration

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub_df = pd.read_csv(SAMPLE_SUB_PATH)
print_initial_exploration(train_df, test_df)

if "target" not in train_df.columns:
    raise ValueError("Expected 'target' column in train.csv")

y = train_df["target"].copy()
X_raw = train_df.drop(columns=["target"]).copy()
X_test_raw = test_df.copy()


## 4. Preprocessing and Feature Engineering

In [ ]:
X = add_feature_engineering(X_raw)
X_test = add_feature_engineering(X_test_raw)

print(f"Feature shape after engineering (train): {X.shape}")
print(f"Feature shape after engineering (test) : {X_test.shape}")

missing_test_cols = sorted(set(X.columns) - set(X_test.columns))
extra_test_cols = sorted(set(X_test.columns) - set(X.columns))

for c in missing_test_cols:
    X_test[c] = np.nan
if extra_test_cols:
    X_test = X_test.drop(columns=extra_test_cols)
X_test = X_test[X.columns]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train split shape: {X_train.shape}, Validation split shape: {X_val.shape}")


## 5. Model 1: Baseline (Linear Regression)

In [ ]:
model1 = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_train, scale_numeric=True)),
        ("regressor", LinearRegression()),
    ]
)
model1.fit(X_train, y_train)
y_pred1 = model1.predict(X_val)
results1 = evaluate_model("Model 1: LinearRegression", y_val, y_pred1)
results1


## 6. Model 2: Advanced (Gradient Boosting + CV Tuning)

In [ ]:
model2_base = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X_train, scale_numeric=False)),
        (
            "regressor",
            GradientBoostingRegressor(
                random_state=RANDOM_STATE,
                n_estimators=400,
                learning_rate=0.05,
                max_depth=3,
                subsample=0.9,
            ),
        ),
    ]
)

cv = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
param_distributions = {
    "regressor__n_estimators": [250, 400, 600, 800],
    "regressor__learning_rate": [0.03, 0.05, 0.08],
    "regressor__max_depth": [2, 3, 4],
    "regressor__subsample": [0.7, 0.85, 1.0],
    "regressor__min_samples_leaf": [1, 3, 5],
}

# RandomizedSearchCV gives a better model than fixed parameters with bounded runtime.
tuner = RandomizedSearchCV(
    estimator=model2_base,
    param_distributions=param_distributions,
    n_iter=12,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=0,
)

tuner.fit(X_train, y_train)
model2 = tuner.best_estimator_
y_pred2 = model2.predict(X_val)
results2 = evaluate_model("Model 2: GradientBoosting", y_val, y_pred2)

print("Best Model 2 params:", tuner.best_params_)
print(f"Best CV RMSE (Model 2): {-tuner.best_score_:.6f}")
results2


## 7. Evaluation and Comparison

In [ ]:
comparison_df = pd.DataFrame([results1, results2]).sort_values("RMSE").reset_index(drop=True)
comparison_df


In [ ]:
winner = comparison_df.iloc[0]
print(
    f"Best model by RMSE: {winner['Model']}. Gradient boosting can capture non-linear interactions and often reduces bias compared with a linear model, while shallow trees and subsampling help control variance."
)

blend_w_model1, blend_rmse = find_best_blend_weight(y_val, y_pred1, y_pred2)
print(f"Blend diagnostic (validation): {blend_w_model1:.2f}*Model1 + {1.0 - blend_w_model1:.2f}*Model2 -> RMSE={blend_rmse:.6f}")

comparison_df.to_csv("model_comparison_metrics.csv", index=False)


## 8. Visualizations for Report

In [ ]:
def plot_prediction_vs_actual(y_true: pd.Series, y_pred: np.ndarray, out_path: str) -> None:
    plt.figure(figsize=(7, 7))
    plt.scatter(y_true, y_pred, alpha=0.5)
    line_min = min(float(np.min(y_true)), float(np.min(y_pred)))
    line_max = max(float(np.max(y_true)), float(np.max(y_pred)))
    plt.plot([line_min, line_max], [line_min, line_max], linestyle="--")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Prediction vs Actual")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_residuals(y_true: pd.Series, y_pred: np.ndarray, out_path: str) -> None:
    residuals = y_true - y_pred
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.5)
    plt.axhline(0.0, linestyle="--")
    plt.xlabel("Predicted")
    plt.ylabel("Residual (Actual - Predicted)")
    plt.title("Residual Plot")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def plot_feature_importance(model2_pipeline: Pipeline, out_path: str, top_n: int = 20) -> None:
    preprocessor = model2_pipeline.named_steps["preprocessor"]
    regressor = model2_pipeline.named_steps["regressor"]
    feature_names = preprocessor.get_feature_names_out()
    importances = regressor.feature_importances_
    order = np.argsort(importances)[::-1][:top_n]
    top_features = np.array(feature_names)[order]
    top_values = importances[order]

    plt.figure(figsize=(10, 7))
    plt.barh(range(len(top_features)), top_values)
    plt.yticks(range(len(top_features)), top_features)
    plt.gca().invert_yaxis()
    plt.xlabel("Importance")
    plt.title(f"Top {top_n} Feature Importances (Model 2)")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


plot_prediction_vs_actual(y_val, y_pred2, "prediction_vs_actual_model2.png")
plot_residuals(y_val, y_pred2, "residual_plot_model2.png")
plot_feature_importance(model2, "feature_importance_model2.png", top_n=20)
print("Saved evaluation artifacts: model_comparison_metrics.csv and 3 plot PNG files")


## 9. Train on Full Data and Create Submission Files

In [ ]:
model1_full = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X, scale_numeric=True)),
        ("regressor", LinearRegression()),
    ]
)

best_params = {k.replace("regressor__", ""): v for k, v in tuner.best_params_.items()}
model2_full = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor(X, scale_numeric=False)),
        ("regressor", GradientBoostingRegressor(random_state=RANDOM_STATE, **best_params)),
    ]
)

model1_full.fit(X, y)
model2_full.fit(X, y)

test_pred1 = np.clip(model1_full.predict(X_test), 0, 100)
test_pred2 = np.clip(model2_full.predict(X_test), 0, 100)

sub1 = sample_sub_df.copy()
sub2 = sample_sub_df.copy()
sub1["target"] = test_pred1
sub2["target"] = test_pred2

sub1.to_csv("submission_model1.csv", index=False)
sub2.to_csv("submission_model2.csv", index=False)
print("Saved submissions: submission_model1.csv, submission_model2.csv")


## 10. Generate Report Text Artifacts

In [ ]:
def build_report_summary(comparison_df: pd.DataFrame) -> str:
    winner = comparison_df.sort_values("RMSE").iloc[0]["Model"]
    lines = [
        "Approach 1: Linear Regression pipeline",
        f"- Why chosen: {APPROACH_NOTES['Model 1: LinearRegression']['why']}",
        f"- How it works: {APPROACH_NOTES['Model 1: LinearRegression']['how']}",
        f"- Drawbacks: {APPROACH_NOTES['Model 1: LinearRegression']['drawbacks']}",
        "",
        "Approach 2: Gradient Boosting Regressor pipeline",
        f"- Why chosen: {APPROACH_NOTES['Model 2: GradientBoosting']['why']}",
        f"- How it works: {APPROACH_NOTES['Model 2: GradientBoosting']['how']}",
        f"- Drawbacks: {APPROACH_NOTES['Model 2: GradientBoosting']['drawbacks']}",
        "",
        "Validation comparison (80/20 split):",
        comparison_df.to_string(index=False),
        "",
        "Metric discussion:",
        "- RMSE is the competition metric and penalizes large errors strongly.",
        "- MAE provides average absolute deviation and is easier to interpret in target units.",
        "- R2 indicates how much target variance is explained by the model.",
        "- MAPE gives relative error proportion but can be unstable near zero targets.",
        "",
        f"Final conclusion: {winner} has the best validation RMSE in this experiment.",
        "Both submissions are clipped to [0, 100] before export.",
    ]
    return "\n".join(lines)


def build_report_draft_markdown(comparison_df: pd.DataFrame) -> str:
    winner = comparison_df.sort_values("RMSE").iloc[0]["Model"]
    markdown = f"""# mlX 2.0 Regression Challenge Report Draft

## 1. Objective
Predict song popularity score (0-100) using only `train.csv` and `test.csv`.

## 2. Data Preparation
- Loaded train/test with pandas.
- Checked shapes, column names, missing values, and dtypes.
- Target variable: `target`.
- Missing value handling:
  - Numerical: median imputation
  - Categorical: constant `Unknown`
- Date processing:
  - Converted `publication_timestamp` to datetime
  - Extracted `pub_year`, `pub_month`, `pub_day`, `pub_weekday`
- Categorical encoding: One-Hot Encoding (`handle_unknown='ignore'`).
- Dropped candidate identifiers/labels: `track_identifier`, `composition_label_0/1/2` (when present).

## 3. Feature Engineering
- Aggregates: `avg_energy`, `avg_danceability`, `avg_tempo`
- Difference: `tempo_range`
- Ratio: `energy_to_danceability`
- Existing derived features such as `emotional_charge` and `groove_efficiency` retained and utilized.

## 4. Modeling Approaches
### Approach 1: Linear Regression
- Why: {APPROACH_NOTES['Model 1: LinearRegression']['why']}
- How: {APPROACH_NOTES['Model 1: LinearRegression']['how']}
- Drawbacks: {APPROACH_NOTES['Model 1: LinearRegression']['drawbacks']}

### Approach 2: Gradient Boosting Regressor
- Why: {APPROACH_NOTES['Model 2: GradientBoosting']['why']}
- How: {APPROACH_NOTES['Model 2: GradientBoosting']['how']}
- Drawbacks: {APPROACH_NOTES['Model 2: GradientBoosting']['drawbacks']}

## 5. Validation Setup and Metrics
- Train/validation split: 80/20 (`random_state={RANDOM_STATE}`).
- Metrics: RMSE (competition metric), MAE, R2, MAPE.

## 6. Results
{comparison_df.to_string(index=False)}

Best by RMSE: **{winner}**

## 7. Metric Discussion
- RMSE emphasizes large errors more than MAE, aligning with leaderboard sensitivity.
- MAE reflects typical absolute prediction error in target units.
- R2 describes variance explained.
- MAPE is useful for relative error but can be unstable for small true values.

## 8. Submission Outputs
- `submission_model1.csv`
- `submission_model2.csv`

## 9. Required Screenshot Placeholders
1. Full-page screenshot of first Kaggle submission proof.
2. Full-page screenshot(s) of final score and leaderboard rank.
3. Any additional full-page screenshots requested by the submission portal.

## 10. Final Conclusion
The stronger non-linear model ({winner}) is preferred for final competition submission in this run.
"""
    return markdown


def build_submission_checklist() -> str:
    return """ASSIGNMENT CHECKLIST

[x] Two different ML approaches implemented
[x] Competition metric (RMSE) computed
[x] Additional metrics computed (MAE, R2, MAPE)
[x] Two submission files generated
[x] Model comparison table produced
[x] Visualizations generated (feature importance, prediction-vs-actual, residuals)
[x] Summary text generated
[ ] First submission full-page screenshot (manual)
[ ] Final score/rank full-page screenshot(s) (manual)
[ ] Final PDF report export (manual)
"""

summary_text = build_report_summary(comparison_df)
report_md = build_report_draft_markdown(comparison_df)
checklist_text = build_submission_checklist()

with open("model_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text + "\n")
with open("report_draft.md", "w", encoding="utf-8") as f:
    f.write(report_md + "\n")
with open("submission_checklist.txt", "w", encoding="utf-8") as f:
    f.write(checklist_text + "\n")

print("Saved text artifacts: model_summary.txt, report_draft.md, submission_checklist.txt")
